# Week 3: Multi-View Reconstruction

This notebook implements the **Incremental SfM Pipeline**.
We loop through a sequence of images, adding them one by one to our 3D map using **PnP**.

In [1]:
import cv2
import numpy as np
import open3d as o3d
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.sfm import SfMMap
from src.reconstruction import get_intrinsic_from_exif, create_point_cloud

In [3]:
# -- 1. Load Image Sequence --

IMAGE_SEQUENCE = [f'photo{i}.jpeg' for i in range(1, 29)]

data_dir = os.path.join(module_path, 'final_ds')
images = []

print(f"Attempting to load {len(IMAGE_SEQUENCE)} images from: {data_dir}")

for name in IMAGE_SEQUENCE:
    path = os.path.join(data_dir, name)
    img = cv2.imread(path)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img_rgb)
        if int(name.replace('photo', '').replace('.jpeg', '')) % 5 == 0:
            print(f"Loaded {name}...")
    else:
        print(f"Warning: Could not load {name} - Check if file exists in data/")

print(f"\nSuccessfully loaded {len(images)} images.")

Attempting to load 28 images from: /Users/emaanwaleed/Documents/GitHub/CS-436---Project/final_ds
Loaded photo5.jpeg...
Loaded photo10.jpeg...
Loaded photo15.jpeg...
Loaded photo20.jpeg...
Loaded photo25.jpeg...

Successfully loaded 28 images.


In [4]:

# -- 2. Initialize Map (Frame 0 & 1) --

first_image_path = os.path.join(data_dir, IMAGE_SEQUENCE[0])

# Estimate K using EXIF data from the file
K = get_intrinsic_from_exif(first_image_path)

print("Intrinsic Matrix K:\n", K)

# Initialize SfM Class
sfm = SfMMap(K)

# Bootstrap with first two images
sfm.initialize(images[0], images[1], lowe_ratio=0.8)

EXIF not found. Using approximation.
Intrinsic Matrix K:
 [[960.   0. 480.]
 [  0. 960. 640.]
 [  0.   0.   1.]]
Initializing Map with first two images...
Found 768 keypoints in img1 and 1315 in img2.
Found 768 initial matches.
Filtered down to 254 good matches using Lowe's ratio test.
Map initialized with 193 points.


In [9]:
# -- 3. Incremental Loop (Frame 2 -> N) --

for i in range(2, len(images)):
    sfm.add_view(images[i])
    
    # Run BA every 5 frames to fix drift
    if i % 5 == 0:
        sfm.refine()

PnP Failed: Geometric check failed.
PnP Failed: Geometric check failed.
View Added. Inliers: 5, New Points: 54, Total: 101
PnP Failed: Geometric check failed.
--- Running Bundle Adjustment on 3 cameras and 101 points ---
   > Setup: 3 cams, 101 points, 207 obs.
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         3.4660e+33                                    1.17e+34    
       1              2         8.8490e+31      3.38e+33       9.95e+08       1.57e+33    
       2              3         2.0218e+28      8.85e+31       1.77e+08       1.74e+31    
       3              4         7.7296e+20      2.02e+28       1.02e+07       2.48e+27    
       4              5         9.6117e+07      7.73e+20       4.93e+06       3.65e+20    
       5              6         9.2938e+07      3.18e+06       2.46e+07       1.16e+19    
       6              8         8.7098e+07      5.84e+06       1.99e+07       3.28e+18    
       7  

In [ ]:
# -- 4. Visualization --

points = np.array(sfm.points_3d)
colors = np.array(sfm.colors)

print(f"Final Cloud has {len(points)} points.")

# Filter outliers 
mean = np.mean(points, axis=0)
std = np.std(points, axis=0)
mask = (np.abs(points - mean) < 2.5 * std).all(axis=1)

filtered_points = points[mask]
filtered_colors = colors[mask]

# Create Point Cloud
pcd = create_point_cloud(filtered_points, filtered_colors)

# --- VISUALIZE TRAJECTORY ---
camera_centers = []
camera_frustums = []

AXIS_SIZE = 2.0 

for R, t in sfm.poses:
    # 4x4 matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    
    # Invert for display (Camera -> World)
    T_inv = np.linalg.inv(T)
    center = T_inv[:3, 3]
    camera_centers.append(center)
    
    axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=AXIS_SIZE, origin=[0,0,0])
    axis.transform(T_inv)
    camera_frustums.append(axis)

# Create a LineSet to connect the camera centers (The Trajectory Path)
if len(camera_centers) > 1:
    lines = []
    for i in range(len(camera_centers) - 1):
        lines.append([i, i+1])
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(camera_centers)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    line_set.colors = o3d.utility.Vector3dVector([[1, 0, 0] for _ in lines])
    
    camera_frustums.append(line_set)

geometries = [pcd] + camera_frustums

print("Opening Visualization...")
print(f"Drawing {len(filtered_points)} points and {len(camera_centers)} cameras.")
o3d.visualization.draw_geometries(geometries, window_name="Week 3: Incremental SfM")

Final Cloud has 101 points.
Opening Visualization...
Drawing 99 points and 3 cameras.


In [ ]:
import json

# -- 5. Export Data for Three.js --

def export_to_json(sfm_map, image_list, output_path="cameras.json"):
    export_data = []
    
    # We assume sfm_map.poses corresponds to image_list indices
    # Note: sfm_map.poses might be slightly shorter if some frames were skipped/failed.
    # However, in your code, we appended a pose for every successful add_view.
    # We need to map them correctly. Ideally, we would have stored the filename with the pose.
    # For now, we assume the first N successful poses match the first N images.
    
    print(f"Exporting {len(sfm_map.poses)} cameras...")
    
    for i, (R, t) in enumerate(sfm_map.poses):
        if i >= len(image_list):
            break
            
        # 1. Convert World->Camera (OpenCV) to Camera->World (Three.js)
        # OpenCV: P_cam = R * P_world + t
        # Three.js needs the matrix that places the camera in the world.
        # This is the INVERSE of [R|t].
        
        # Construct 4x4 matrix
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = t.flatten()
        
        # Invert it
        T_inv = np.linalg.inv(T)
        
        # 2. Extract Position and Rotation (Quaternion)
        # We can just save the matrix, Three.js handles it easily.
        # Flatten to column-major list for JS compatibility if needed, 
        # or just save as row-major 4x4 array.
        
        cam_dict = {
            "id": i,
            "filename": image_list[i], # We assume order is preserved
            "matrix": T_inv.flatten().tolist() # Flattened 4x4 matrix
        }
        export_data.append(cam_dict)

    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=4)
    
    print(f"Saved {output_path}")

# Run the export
# Make sure 'images' list from earlier cells matches the sequence used in SfM
# If you used 'IMAGE_SEQUENCE' to load names, pass that.
export_to_json(sfm, IMAGE_SEQUENCE, output_path="cameras.json")

# Also save the PLY if you haven't already
o3d.io.write_point_cloud("point_cloud.ply", pcd)
print("Saved point_cloud.ply")

Exporting 35 cameras...
Saved cameras.json
Saved point_cloud.ply


In [ ]:
# Week 3: FAST VERSION (No Bundle Adjustment, Quick Reconstruction)

import cv2
import numpy as np
import open3d as o3d
import os
import sys
import json

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.sfm import SfMMap
from src.reconstruction import get_intrinsic_from_exif, create_point_cloud

# Configuration
NUM_IMAGES = 20  # Start with 20 images
LOWE_RATIO = 0.75  # Matching threshold

# -- 1. Load Images --
IMAGE_SEQUENCE = [f'photo{i}.jpeg' for i in range(1, NUM_IMAGES + 1)]
data_dir = os.path.join(module_path, 'data2')
images = []

print(f"Loading {len(IMAGE_SEQUENCE)} images...")
for name in IMAGE_SEQUENCE:
    path = os.path.join(data_dir, name)
    img = cv2.imread(path)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img_rgb)

print(f"✓ Loaded {len(images)} images\n")

# -- 2. Initialize --
first_image_path = os.path.join(data_dir, IMAGE_SEQUENCE[0])
K = get_intrinsic_from_exif(first_image_path)

sfm = SfMMap(K)

print("Initializing with first two images...")
sfm.initialize(images[0], images[1], lowe_ratio=LOWE_RATIO)
print(f"✓ Initial points: {len(sfm.points_3d)}\n")

# -- 3. Add Views (FAST - No BA) --
print("Adding remaining views...")
for i in range(2, len(images)):
    try:
        success = sfm.add_view(images[i])
        status = "✓" if success else "✗"
        print(f"{status} View {i:2d}: {len(sfm.points_3d):5d} points, {len(sfm.poses):2d} cams")
    except Exception as e:
        print(f"✗ View {i:2d}: ERROR - {str(e)[:50]}")

# -- 4. Filter and Visualize --
print(f"\n{'='*60}")
print("FILTERING OUTLIERS...")
print(f"{'='*60}")

points = np.array(sfm.points_3d)
colors = np.array(sfm.colors)

print(f"Before: {len(points)} points")

# Simple percentile-based filtering
percentile_low = np.percentile(points, 5, axis=0)
percentile_high = np.percentile(points, 95, axis=0)

mask = (
    (points[:, 0] >= percentile_low[0]) & (points[:, 0] <= percentile_high[0]) &
    (points[:, 1] >= percentile_low[1]) & (points[:, 1] <= percentile_high[1]) &
    (points[:, 2] >= percentile_low[2]) & (points[:, 2] <= percentile_high[2])
)

filtered_points = points[mask]
filtered_colors = colors[mask]

print(f"After: {len(filtered_points)} points")

# Create point cloud
pcd = create_point_cloud(filtered_points, filtered_colors)

# Camera visualization
camera_centers = []
for R, t in sfm.poses:
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    T_inv = np.linalg.inv(T)
    camera_centers.append(T_inv[:3, 3])

# Camera trajectory line
geometries = [pcd]
if len(camera_centers) > 1:
    lines = [[i, i+1] for i in range(len(camera_centers)-1)]
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(camera_centers)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    line_set.colors = o3d.utility.Vector3dVector([[1, 0, 0] for _ in lines])
    geometries.append(line_set)

# -- 5. Export --
print(f"\n{'='*60}")
print("EXPORTING...")
print(f"{'='*60}")

# Export cameras
cameras_export = []
for i, (R, t) in enumerate(sfm.poses):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    T_inv = np.linalg.inv(T)
    
    cameras_export.append({
        "id": i,
        "filename": IMAGE_SEQUENCE[i] if i < len(IMAGE_SEQUENCE) else f"unknown_{i}",
        "matrix": T_inv.flatten().tolist()
    })

with open("cameras.json", 'w') as f:
    json.dump(cameras_export, f, indent=2)

o3d.io.write_point_cloud("point_cloud.ply", pcd)

print(f"✓ Saved cameras.json ({len(cameras_export)} cameras)")
print(f"✓ Saved point_cloud.ply ({len(filtered_points)} points)")

# -- 6. Summary --
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total images attempted: {len(images)}")
print(f"Successful cameras: {len(sfm.poses)}")
print(f"3D points reconstructed: {len(filtered_points)}")
print(f"Success rate: {len(sfm.poses)/len(images)*100:.1f}%")

# Check if reconstruction is reasonable
if len(filtered_points) < 500:
    print("\n⚠️  WARNING: Very sparse reconstruction!")
    print("   Possible issues:")
    print("   - Not enough image overlap")
    print("   - Try adjusting LOWE_RATIO (currently {})".format(LOWE_RATIO))
    print("   - Check if images are sequential/continuous")
elif len(sfm.poses) < len(images) * 0.5:
    print("\n⚠️  WARNING: Many views failed to add!")
    print("   Consider checking image quality/overlap")
else:
    print("\n✓ Reconstruction looks reasonable!")

print(f"\n{'='*60}")
print("OPENING 3D VIEWER...")
print(f"{'='*60}")

o3d.visualization.draw_geometries(
    geometries,
    window_name="Week 3 Reconstruction (Fast)",
    width=1280,
    height=720
)

Loading 20 images...
✓ Loaded 20 images

EXIF not found. Using approximation.
Initializing with first two images...
Initializing Map with first two images...
Found 2483 keypoints in img1 and 2679 in img2.
Found 2483 initial matches.
Filtered down to 541 good matches using Lowe's ratio test.
Map initialized with 283 points.
✓ Initial points: 283

Adding remaining views...
View Added. Inliers: 6, New Points: 371, Total: 654
✗ View  2:   654 points,  3 cams
View Added. Inliers: 12, New Points: 337, Total: 991
✗ View  3:   991 points,  4 cams
View Added. Inliers: 6, New Points: 397, Total: 1388
✗ View  4:  1388 points,  5 cams
View Added. Inliers: 65, New Points: 470, Total: 1858
✗ View  5:  1858 points,  6 cams
View Added. Inliers: 142, New Points: 194, Total: 2052
✗ View  6:  2052 points,  7 cams
View Added. Inliers: 206, New Points: 220, Total: 2272
✗ View  7:  2272 points,  8 cams
View Added. Inliers: 232, New Points: 223, Total: 2495
✗ View  8:  2495 points,  9 cams
PnP Failed: Geomet

In [7]:
from PIL import Image
from PIL.ExifTags import TAGS

img = Image.open(r"/Users/emaanwaleed/Documents/GitHub/CS-436---Project/final_ds/photo1.jpeg")
exif = img._getexif()
print(exif)


None
